In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Fix: avoid imports that can trigger protobuf-related crashes in this environment.
# (cv2/seaborn/matplotlib are not needed for training/inference here.)

# Reproducibility/stability
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("Keras:", keras.__version__)



2026-05-14 09:41:02.054632: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778751662.067607      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778751662.071602      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-14 09:41:02.087035: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF: 2.18.0
Keras: 3.8.0


In [2]:
# Paths (keep Kaggle-style paths)
BASE = "/kaggle/input/cassava-leaf-disease-classification"
TRAIN_CSV = os.path.join(BASE, "train.csv")
SAMPLE_SUB = os.path.join(BASE, "sample_submission.csv")
TRAIN_IMG_DIR = os.path.join(BASE, "train_images")
TEST_IMG_DIR = os.path.join(BASE, "test_images")

assert os.path.exists(TRAIN_CSV), f"Missing {TRAIN_CSV}"
assert os.path.exists(SAMPLE_SUB), f"Missing {SAMPLE_SUB}"
assert os.path.exists(TRAIN_IMG_DIR), f"Missing {TRAIN_IMG_DIR}"
assert os.path.exists(TEST_IMG_DIR), f"Missing {TEST_IMG_DIR}"

train_df = pd.read_csv(TRAIN_CSV)
ss = pd.read_csv(SAMPLE_SUB)

print(train_df.shape, ss.shape)
print(train_df.head())



(18721, 2) (2676, 2)
         image_id  label
0  1000015157.jpg      0
1  1000201771.jpg      3
2   100042118.jpg      1
3  1000723321.jpg      1
4  1000812911.jpg      3


In [3]:
# Image pipeline
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 5
AUTOTUNE = tf.data.AUTOTUNE


def decode_and_resize(path, label=None):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE, method="bilinear", antialias=True)
    img = tf.cast(img, tf.float32) / 255.0
    if label is None:
        return img
    return img, tf.cast(label, tf.int32)


# Train/val split (no leakage)
train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_frac = 0.1
val_n = int(len(train_df) * val_frac)
val_df = train_df.iloc[:val_n].copy()
tr_df = train_df.iloc[val_n:].copy()

tr_paths = (TRAIN_IMG_DIR + "/" + tr_df["image_id"]).values
tr_labels = tr_df["label"].values
val_paths = (TRAIN_IMG_DIR + "/" + val_df["image_id"]).values
val_labels = val_df["label"].values

train_ds = tf.data.Dataset.from_tensor_slices((tr_paths, tr_labels))
train_ds = train_ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
train_ds = (
    train_ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)



2026-05-14 09:41:08.797742: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778751668.798159      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:05:00.0, compute capability: 8.6


In [4]:
# Fix: model file doesn't exist; build a small CNN model consistent with the intended CNN classifier core logic.
inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = keras.layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
x = keras.layers.MaxPooling2D()(x)
x = keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = keras.layers.MaxPooling2D()(x)
x = keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = keras.layers.GlobalAveragePooling2D()(x)

# "MLP head"
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,405 (431.27 KB)

 Trainable params: 110,405 (431.27 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Train (kept straightforward; no early stopping / no sampling)
EPOCHS = (
    5  # chosen to fit within runtime; core logic remains standard supervised training
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=2,
)



Epoch 1/5


I0000 00:00:1778751672.983524     106 service.cc:148] XLA service 0x7fc81c004bf0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778751672.983557     106 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-14 09:41:13.023858: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778751673.137044     106 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-14 09:41:14.917465: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_962', 60 bytes spill stores, 64 bytes spill loads

I0000 00:00:1778751676.473096     106 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-05-14 09:41:31.735578: I external/local_xla/x

527/527 - 27s - 51ms/step - accuracy: 0.6168 - loss: 1.1461 - val_accuracy: 0.6309 - val_loss: 1.0707
Epoch 2/5
527/527 - 14s - 27ms/step - accuracy: 0.6301 - loss: 1.0580 - val_accuracy: 0.6314 - val_loss: 1.0669
Epoch 3/5
527/527 - 15s - 28ms/step - accuracy: 0.6388 - loss: 1.0123 - val_accuracy: 0.6378 - val_loss: 0.9835
Epoch 4/5
527/527 - 16s - 30ms/step - accuracy: 0.6440 - loss: 0.9827 - val_accuracy: 0.6485 - val_loss: 0.9732
Epoch 5/5
527/527 - 15s - 29ms/step - accuracy: 0.6475 - loss: 0.9675 - val_accuracy: 0.6410 - val_loss: 0.9555


In [6]:
# Predict on test set in the sample_submission order and write a valid submission.csv
test_paths = (TEST_IMG_DIR + "/" + ss["image_id"]).values
test_ds = tf.data.Dataset.from_tensor_slices(test_paths)
test_ds = test_ds.map(lambda p: decode_and_resize(p, None), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

probs = model.predict(test_ds, verbose=1)
preds = np.argmax(probs, axis=1).astype(int)

my_submission = pd.DataFrame({"image_id": ss["image_id"].values, "label": preds})
my_submission.to_csv("submission.csv", index=False)

print(my_submission.head())
print("Wrote submission.csv with shape:", my_submission.shape)
assert my_submission.shape[0] == ss.shape[0]
assert list(my_submission.columns) == ["image_id", "label"]
assert os.path.exists("submission.csv")

84/84 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      3
3  1234555380.jpg      3
4  1234571117.jpg      3
Wrote submission.csv with shape: (2676, 2)
